In [1]:
%pip install mysql-connector-python
import pandas as pd
from sqlalchemy import create_engine, URL, text

database_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password="Nathandrake9611",
    host="localhost",
    port=3306,
    database="sakila"
)

engine = create_engine(database_url)

Note: you may need to restart the kernel to use updated packages.


In [2]:
try:
    with engine.connect() as connection:
        database_name = connection.execute(
            text("SELECT DATABASE();")
        ).scalar()

        print(f"Successfully connected to: {database_name}")

except Exception as error:
    print(f"Connection error: {error}")

Successfully connected to: sakila


In [3]:
import pandas as pd
from sqlalchemy import text


def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    """
    Retrieve Sakila rental records for a specified month and year.

    Parameters
    ----------
    engine:
        SQLAlchemy database engine connected to Sakila.
    month:
        Month number from 1 to 12.
    year:
        Four-digit year.

    Returns
    -------
    pd.DataFrame
        Rental records matching the specified month and year.
    """

    if not 1 <= month <= 12:
        raise ValueError("month must be an integer between 1 and 12")

    query = text("""
        SELECT
            rental_id,
            rental_date,
            inventory_id,
            customer_id,
            return_date,
            staff_id
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year
        ORDER BY rental_date;
    """)

    return pd.read_sql(
        query,
        con=engine,
        params={
            "month": month,
            "year": year
        }
    )

In [4]:
may_rentals = rentals_month(engine, 5, 2005)

may_rentals.head()


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1


In [5]:
def rentals_count_month(rentals_df: pd.DataFrame, month: int, year: int) -> pd.DataFrame:

    if not 1 <= month <= 12:
        raise ValueError("month must be an integer between 1 and 12")

    if not isinstance(year, int):
        raise TypeError("year must be an integer")

    # Filter the DataFrame for the specified month and year
    filtered_df = rentals_df[
        (rentals_df["rental_date"].dt.month == month) &
        (rentals_df["rental_date"].dt.year == year)
    ]

    # Count the number of rentals for each customer
    rentals_count = filtered_df.groupby("customer_id").size()

    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        "customer_id": rentals_count.index,
        "rentals_count": rentals_count.values
    })

    return result_df

In [11]:
rent_tot = rentals_count_month(may_rentals, 5, 2005)
rent_tot

,customer_id,rentals_count
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3
...,...,...
515,594,4
516,595,1
517,596,6
518,597,2


In [12]:
def compare_rentals(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:

    rental_col1 = [col for col in df1.columns if col != "customer_id"][0]
    rental_col2 = [col for col in df2.columns if col != "customer_id"][0]

    combined_df = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="inner"
    )

    combined_df["difference"] = (
        combined_df[rental_col2] - combined_df[rental_col1]
    )

    return combined_df

In [13]:
comparison = compare_rentals(may_count, june_count)
comparison.head()

NameError: name 'may_count' is not defined